In [ ]:
    ############    #############   SOLID and Clean Architecture   #############   ##############   

 =>  SOLID is five principles for keeping a codebase changeable as it grows -- each one is
       really about limiting how far a change in one place forces changes everywhere else.

 =>  Clean architecture (and the layered router/service/repository pattern from Phase 0.2)
       is SOLID applied at the module/layer level: dependencies point inward, toward
       abstractions, not outward toward frameworks or databases.


<img src="images/solid-at-a-glance.png" alt="SOLID at a glance: Single Responsibility, Open/Closed, Liskov Substitution, Interface Segregation, Dependency Inversion">

In [ ]:
# --- Violates Single Responsibility: this class both computes AND formats/sends ---
class BadInvoice:
    def __init__(self, items: list[float]):
        self.items = items

    def total(self) -> float:
        return sum(self.items)

    def print_receipt(self) -> None:   # a second, unrelated reason to change
        print(f"Total due: ${self.total():.2f}")


# --- Fixed: one class computes, another formats ---
class Invoice:
    def __init__(self, items: list[float]):
        self.items = items

    def total(self) -> float:
        return sum(self.items)

class ReceiptPrinter:
    def print_receipt(self, invoice: Invoice) -> None:
        print(f"Total due: ${invoice.total():.2f}")

invoice = Invoice([19.99, 5.00])
ReceiptPrinter().print_receipt(invoice)


In [ ]:
 =>  Now 'how a receipt looks' can change (e.g. add a PDF export) without ever touching
       Invoice, and 'how a total is computed' can change (e.g. add tax) without touching
       ReceiptPrinter -- each class has exactly one reason to change.


In [ ]:
from abc import ABC, abstractmethod

# --- Dependency Inversion: depend on an abstraction, not a concrete notifier ---
class Notifier(ABC):
    @abstractmethod
    def send(self, message: str) -> None: ...

class EmailNotifier(Notifier):
    def send(self, message: str) -> None:
        print(f"[email] {message}")

class SlackNotifier(Notifier):   # Open/Closed: adding this required ZERO changes above
    def send(self, message: str) -> None:
        print(f"[slack] {message}")

class OrderService:
    def __init__(self, notifier: Notifier):  # depends on the abstraction
        self._notifier = notifier

    def place_order(self, order_id: int) -> None:
        self._notifier.send(f"order {order_id} placed")

OrderService(EmailNotifier()).place_order(1)
OrderService(SlackNotifier()).place_order(2)


In [ ]:
 =>  OrderService never imports EmailNotifier or SlackNotifier directly -- it only knows
       about the Notifier abstraction. Adding a third channel (SMS, webhook) means writing a
       new class, not modifying OrderService (Open/Closed) -- and any Notifier subclass is
       safely substitutable anywhere a Notifier is expected (Liskov Substitution).


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Find one class in a real project (yours or open source) that violates Single
           Responsibility. Split it and confirm existing tests still pass.

 =>  [ ] Write an Interface Segregation example: split one 'fat' interface (e.g. a Worker
           with print(), scan(), fax(), email() methods) into smaller, role-specific ones.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Applying SOLID dogmatically to trivial code -- a 10-line script doesn't need an
       abstract Notifier hierarchy; these principles pay off as complexity/change grows.

 =>  Confusing 'Open/Closed' with 'never touch this file again' -- it means closed to
       modification for NEW variants of existing behavior, not frozen forever.
